<table style="width:100%">
<tr>
<td style="vertical-align:middle; text-align:left;">
<font size="2">
Supplementary code for the <a href="http://mng.bz/orYv">Build a Large Language Model From Scratch</a> book by <a href="https://sebastianraschka.com">Sebastian Raschka</a><br>
<br>Code repository: <a href="https://github.com/rasbt/LLMs-from-scratch">https://github.com/rasbt/LLMs-from-scratch</a>
</font>
</td>
<td style="vertical-align:middle; text-align:left;">
<a href="http://mng.bz/orYv"><img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/cover-small.webp" width="100px"></a>
</td>
</tr>
</table>

# 使用Llama 3.1 70B和Ollama生成偏好数据集

- 偏好微调是一个将指令微调的LLM与人类偏好对齐的过程
- 存在多种方法来创建用于偏好微调LLM的数据集
  1. 我们使用指令微调的 LLM 生成多个响应,并让人类根据其偏好和/或给定的偏好标准对它们进行排序
  2. 我们使用指令微调的 LLM 生成多个响应，并由 LLMs 根据给定的偏好标准对它们进行排序
  3. 我们使用 LLM 根据特定的偏好标准生成偏好和不受偏好的响应
- 在这个笔记本中，我们考虑方法 3
- 这个笔记本通过 ollama 使用一个拥有 700 亿参数的 Llama 3.1-Instruct 模型来为指令数据集生成偏好标签
- 指令数据集的预期格式如下：


### Input

```json
[
    {
        "instruction": "What is the state capital of California?",
        "input": "",
        "output": "The state capital of California is Sacramento.",
    },
    {
        "instruction": "Provide a synonym for 'fast'.",
        "input": "",
        "output": "A synonym for 'fast' is 'quick'.",
    },
    {
        "instruction": "What is the capital of Greece?",
        "input": "",
        "output": "The capital of Greece is Athens.",

    },
...
]
```

输出数据集将如下所示，其中更礼貌的回应更受偏好 (`'chosen'`), 而不礼貌的回答不受偏好 (`'rejected'`):

```json
[
    {
        "instruction": "What is the state capital of California?",
        "input": "",
        "output": "The state capital of California is Sacramento.",
        "rejected": "Look, the state capital of California is obviously Sacramento.",
        "chosen": "The state capital of California is Sacramento."
    },
    {
        "instruction": "Provide a synonym for 'fast'.",
        "input": "",
        "output": "A synonym for 'fast' is 'quick'.",
        "chosen": "A suitable alternative to 'fast' would be 'quick'.",
        "rejected": "A synonym for 'fast' is 'quick'."
    },
    {
        "instruction": "What is the capital of Greece?",
        "input": "",
        "output": "The capital of Greece is Athens.",
        "chosen": "I'd be happy to help! The capital of Greece is indeed Athens.",
        "rejected": "The capital of Greece is Athens."
    },
...
]
```

### Output




- 代码不需要GPU，只要足够的RAM就可以在笔记本电脑上运行

In [ ]:
from importlib.metadata import version

pkgs = ["tqdm",    # Progress bar
        ]

for p in pkgs:
    print(f"{p} version: {version(p)}")

tqdm version: 4.67.3


## Installing Ollama and Downloading Llama 3.1

- Ollama 是一个用于高效运行 LLMs 的应用程序
- 它是一个围绕[llama.cpp](https://github.com/ggerganov/llama.cpp)的封装, 该封装使用存C/C++实现LLMs以最大化效率
- 请注意，它是一个用于使用 LLMs 生成文本（推理）的工具，而不是用于训练或微调 LLMs 的工具
- 在运行下方代码之前，请访问[https://ollama.com](https://ollama.com) 并按照说明安装 ollama（例如，点击“下载”按钮并下载适用于您操作系统的 ollama 应用程序）

- 对于 macOS 和 Windows 用户，点击您下载的 ollama 应用程序；如果提示您安装命令行使用，请回答“是”
- Linux 用户可以使用 ollama 网站上提供的安装命令

- 一般来说，在命令行中使用 ollama 之前，我们必须要么启动 ollama 应用程序，要么在一个单独的终端中运行 ollama serve

<img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/bonus/ollama-eval/ollama-serve.webp?1">


- 在使用 ollama 应用或 ollama serve 时，在另一个终端中，通过命令行执行以下命令来试用 70 亿参数的 Llama 3.1 模型 

```bash
# 70B model
ollama run llama3.1:70b
```


输出如下:

```
$ ollama run llama3.1:70b
pulling manifest
pulling aa81b541aae6... 100% ▕████████████████▏ 39 GB
pulling 8cf247399e57... 100% ▕████████████████▏ 1.7 KB
pulling f1cd752815fc... 100% ▕████████████████▏ 12 KB
pulling 56bb8bd477a5... 100% ▕████████████████▏ 96 B
pulling 3c1c2d3df5b3... 100% ▕████████████████▏ 486 B
verifying sha256 digest
writing manifest
removing any unused layers
success
```

- 注意，`llama3.1：70b` 指的是经过指令微调的 700 亿参数 Llama 3.1 模型

- 或者，你也可以使用更小、资源更高效的 80 亿参数 Llama 3.1 模型，通过将  `llama3.1:70b` 替换成 `llama3.1`

- 下载完成后，你会看到一个命令行提示符，允许你与模型聊天

- 试试类似“骆马吃什么？”这样的提示，应该会返回类似以下内容的输出：

```
>>> What do llamas eat?
Llamas are ruminant animals, which means they have a four-chambered 
stomach and eat plants that are high in fiber. In the wild, llamas 
typically feed on:
1. Grasses: They love to graze on various types of grasses, including tall 
grasses, wheat, oats, and barley.
```

- 你可以输入 `/bye` 结束本次会话

## Using Ollama's REST API

- 另一种交互方式是通过 Python 中的 REST API，通过以下函数
- 在运行本笔记本的下一个单元之前，确保 ollama 仍在运行，如上所述，通过以下方式
  - `ollama serve` 在终端
  - Ollama 应用
- 接下来，运行以下代码单元来查询模型

- 首先，让我们用一个简单的例子来验证 API，确保它能按预期工作：

In [ ]:
import json
import requests


def query_model(prompt, model="llama3.1", url="http://localhost:11434/api/chat"):
    # 创建了一个data字典,其中包含AI需要的指令
    data = {
        "model": model, #模型名称
        "messages": [
            {
                "role": "user", #提示词来自用户
                "content": prompt 
            }
        ],
        "options": { #选项设置使AI的回答非常一致和可预测
            "seed": 123,
            "temperature": 0,
        }
    }

    # 使用POST请求将data发送到url
    with requests.post(url, json=data, stream=True, timeout=30) as r:
        r.raise_for_status() #检查请求是否成功
        response_data = "" 
        
        #将答案拼凑起来,遍历服务器每一行
        for line in r.iter_lines(decode_unicode=True):
            if not line: #忽略空行
                continue
            response_json = json.loads(line) #将该行从文本转回json字典
            if "message" in response_json:
                #从消息的content部分提取实际内容太并将它们添加到response_data中
                response_data += response_json["message"]["content"]

    return response_data


result = query_model("What do Llamas eat?")
print(result)

Llamas are herbivores, which means they primarily eat plants and plant-based foods. Their diet consists of:

1. **Grasses**: They love to graze on various types of grasses, including tall fescue, orchard grass, and bluegrass.
2. **Hay**: Timothy hay, alfalfa hay, and other types of hay are staples in a llama's diet.
3. **Fruits**: Apples, carrots, and sweet potatoes are all treats that llamas enjoy.
4. **Vegetables**: Leafy greens like kale, spinach, and collard greens are also part of their diet.
5. **Grains**: Oats, corn, and barley can be given to llamas as a supplement or treat.

In the wild, llamas would typically eat whatever plants are available in their native Andean habitats, including shrubs, leaves, and flowers.

As pets or on farms, llama owners often provide a balanced diet that includes:

* 50-60% hay ( Timothy hay or alfalfa)
* 20-30% grass
* 10-20% grains (oats, corn, barley)
* 5-10% fruits and vegetables

It's essential to note that llamas have specific dietary needs, 

## Load JSON Entries

- 现在，让我们进入数据生成部分
- 这里，作为一个实际操作的例子，我们使用了最初在第 7 章用来进行指令微调模型的 `instruction-data.json` 文件：

In [ ]:
from pathlib import Path

json_file = Path("..", "01_main-chapter-code", "instruction-data.json")

with open(json_file, "r") as file:
    json_data = json.load(file)

print("Number of entries:", len(json_data))

Number of entries: 1100


- 该文件的结构如下，我们有测试数据集中给定的响应 (`'output'`) 通过基于 `'input'` 和 `'instruction'`进行指令微调训练模型生成

In [ ]:
json_data[0]

{'instruction': 'Evaluate the following phrase by transforming it into the spelling given.',
 'input': 'freind --> friend',
 'output': 'The spelling of the given phrase "freind" is incorrect, the correct spelling is "friend".'}

- 下面是一个小型实用函数，用于格式化指令和输入：

In [ ]:
#把字典格式的数据包含 instruction 指令 和 input 补充输入），拼接成一段连贯的文本。
def format_input(entry):
    instruction_text = (
        #以下是描述任务的指令,请编写一个恰当完成该请求的回复(告诉AI它现在的角色是什么)
        f"Below is an instruction that describes a task. Write a response that "
        f"appropriately completes the request."
        f"\n\n### Instruction:\n{entry['instruction']}" #
    )

    #如果entry中存在input内容,就拼接上### Input:标签,如果input为空,就直接等于一个空字符串
    input_text = f"\n\n### Input:\n{entry['input']}" if entry["input"] else ""
    instruction_text + input_text

    return instruction_text + input_text

"""举例
entry = {
    "instruction": "请把这句话翻译成英文。",
    "input": "我喜欢学习编程。"
}

Below is an instruction that describes a task. Write a response that appropriately completes the request.

### Instruction:
请把这句话翻译成英文。

### Input:
我喜欢学习编程。
"""

- 现在，让我们试试 ollama API 生成 `'chosen'` 和 `'rejected'` 的响应, 用于偏好调整模型
- H在这里，为了说明方便，我们会给出或多或少礼貌的回答


In [ ]:
import random


for entry in json_data[:5]:
    
    #随机挑选语气
    politeness = random.choice(["polite", "impolite"])    
    #构建AI提示词
    '''
    给定输入内容 { 这里调用了您刚才问过的 format_input 函数组装输入 }，以及正确的输出内容 { 原本的标准答案 }。
    请稍微改写这个输出，使其变得更加 { 这里填入刚才随机选的 polite 或 impolite }。
    请保持修改幅度尽可能小。
    仅仅返回你生成的回复，不要包含任何其他废话。”
    '''
    prompt = (
        f"Given the input `{format_input(entry)}` "
        f"and correct output `{entry['output']}`, "
        f"slightly rewrite the output to be more {politeness}."
        "Keep the modification minimal."
        "Only return return the generated response and nothing else."
    )
    print("\nDataset response:")
    print(">>", entry['output'])
    print(f"\n{politeness} response:")
    print(">>", query_model(prompt))    


Dataset response:
>> The spelling of the given phrase "freind" is incorrect, the correct spelling is "friend".

impolite response:
>> The spelling of the given phrase "freind" is utterly wrong, the correct spelling is "friend".

Dataset response:
>> He goes to the park every day.

polite response:
>> He tends to go to the park every day.

Dataset response:
>> 45 kilometers is 45000 meters.

polite response:
>> You are correct, 45 kilometers is equivalent to 45,000 meters.

Dataset response:
>> Although it was raining, they went for a walk.

impolite response:
>> Although it was pouring down rain, they still decided to go for a bloody walk.

Dataset response:
>> 1, 4, 9, 16, 25, 36, 49, 64, 81, 100.

impolite response:
>> Here are the first 10 square numbers: 1, 4, 9, 16, 25, 36, 49, 64, 81, 100. Don't bother asking for more, you're not that interested anyway.


- 如果我们发现上述生成的回答看起来合理，就可以进入下一步，将提示应用于整个数据集
- 这里，我们为首选回答添加 `'chosen'` 键，为不偏好的回答添加 `'rejected'` 键

In [ ]:
import random
from tqdm import tqdm

def generate_model_responses(json_data):
    
    #将json_data包装在一个进度条中,并在进度条旁边显示"Writing entries"的提示文字
    for i, entry in enumerate(tqdm(json_data, desc="Writing entries")):
        politeness = random.choice(["polite", "impolite"])    
        prompt = (
            f"Given the input `{format_input(entry)}` "
            f"and correct output `{entry['output']}`, "
            f"slightly rewrite the output to be more {politeness}."
            "Keep the modification minimal."
            "Only return return the generated response and nothing else."
        )
        #发送给大模型,将结果存在response中
        response = query_model(prompt)
        
        if politeness == "polite":
            json_data[i]["chosen"] = response #将AI生成的礼貌回复作为好的回复
            json_data[i]["rejected"] = entry["output"] #将原始回复作为较差的回复
        else:
            json_data[i]["rejected"] = response #将AI生成的不礼貌回复作为拒绝回复
            json_data[i]["chosen"] = entry["output"]#将原始标准回复作为较好回复


- 现在我们将此评估应用于整个数据集，计算每个模型的平均得分（在 M3 MacBook Air 笔记本上，每个模型大约需要 1 分钟）
- 请注意，ollama 在各操作系统之间并非完全确定性（截至本文撰写时），因此你得到的数值可能与下面显示的略有差异

In [13]:
generate_model_responses(json_data)

Writing entries:  52%|█████▏    | 576/1100 [39:18<35:45,  4.09s/it]  


ReadTimeout: HTTPConnectionPool(host='127.0.0.1', port=7897): Read timed out. (read timeout=30)

In [ ]:
with open("instruction-data-with-preference.json", "w") as file:
    json.dump(json_data, file, indent=4)